In [1]:
!pip install requests

# 1. Get access token

In [2]:
import requests

CLIENT_ID = "sh-f359ad3d-e4d1-4998-a1d4-b91faa4e5b8f"
CLIENT_SECRET = "i0jOjKpBCV06gCKoDddbe6MA1UCLmFjv"

url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"

data = {
    "grant_type": "client_credentials",
    "client_id": CLIENT_ID,
    "client_secret": CLIENT_SECRET
}

response = requests.post(url, data=data)
token = response.json()
ACCESS_TOKEN = token["access_token"]

print("Token OK")
print(token)

Token OK
{'access_token': 'eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJYVUh3VWZKaHVDVWo0X3k4ZF8xM0hxWXBYMFdwdDd2anhob2FPLUxzREZFIn0.eyJleHAiOjE3NTgxMjUyMjEsImlhdCI6MTc1ODEyNDYyMSwianRpIjoiMmY1ZDI4YTQtOWNhNS00MzcxLWE4ZjAtNDJiOGY2NzBhYTQ2IiwiaXNzIjoiaHR0cHM6Ly9pZGVudGl0eS5kYXRhc3BhY2UuY29wZXJuaWN1cy5ldS9hdXRoL3JlYWxtcy9DRFNFIiwic3ViIjoiOTVmYjJiZTMtZWI5NC00OGViLWE5MWMtMDg1MDNlZmVhMjU0IiwidHlwIjoiQmVhcmVyIiwiYXpwIjoic2gtZjM1OWFkM2QtZTRkMS00OTk4LWExZDQtYjkxZmFhNGU1YjhmIiwic2NvcGUiOiJlbWFpbCBwcm9maWxlIHVzZXItY29udGV4dCIsImNsaWVudEhvc3QiOiIxMTYuMTEwLjQwLjEyMSIsImVtYWlsX3ZlcmlmaWVkIjpmYWxzZSwib3JnYW5pemF0aW9ucyI6WyJkZWZhdWx0LWFkNTIyY2I5LTA5MWQtNDJlZS1iOTc2LTJlYjU2MzZiMTNjNyJdLCJ1c2VyX2NvbnRleHRfaWQiOiI4ZjFmYWVmOC00NDRmLTRhNzItYTJkYi00ZTNmY2RiYTFlZWQiLCJjb250ZXh0X3JvbGVzIjp7fSwiY29udGV4dF9ncm91cHMiOlsiL2FjY2Vzc19ncm91cHMvdXNlcl90eXBvbG9neS9jb3Blcm5pY3VzX2dlbmVyYWwvIiwiL29yZ2FuaXphdGlvbnMvZGVmYXVsdC1hZDUyMmNiOS0wOTFkLTQyZWUtYjk3Ni0yZWI1NjM2YjEzYzcvIl0sInByZWZlcnJlZF91c2VybmFtZSI6InNlcnZpY

# 2. List the latest 3 products in the catalogue

In [3]:
headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}

url = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
params = {"$top": 3, "$orderby": "ContentDate/Start desc"}

r = requests.get(url, headers=headers, params=params)
results = r.json()

for item in results["value"]:
    print("Name:", item["Name"])
    print("PublicationDate:", item["PublicationDate"])
    print("Size MB:", round(item["ContentLength"]/1e6, 2))
    print("Online:", item["Online"])
    print("---")

Name: S5P_NRTI_L1B_RA_BD6_20250917T152323_20250917T152835_41090_03_020101_20250917T154406.nc
PublicationDate: 2025-09-17T15:53:39.487991Z
Size MB: 692.37
Online: True
---
Name: S5P_NRTI_L1B_RA_BD5_20250917T152323_20250917T152835_41090_03_020101_20250917T154406.nc
PublicationDate: 2025-09-17T15:53:38.262116Z
Size MB: 692.37
Online: True
---
Name: S5P_NRTI_L1B_RA_BD7_20250917T152323_20250917T152835_41090_03_020101_20250917T154406.nc
PublicationDate: 2025-09-17T15:53:26.128937Z
Size MB: 319.24
Online: True
---


# 3. Filter by mission (e.g., Sentinel-2)

In [4]:
params = {
    "$top": 5,
    "$orderby": "ContentDate/Start desc",
    "$filter": "Collection/Name eq 'SENTINEL-2'"
}
r = requests.get(url, headers=headers, params=params)
results = r.json()

for item in results["value"]:
    print("Sentinel-2 product:", item["Name"], "| Date:", item["OriginDate"])

Sentinel-2 product: S2C_MSIL2A_20250917T110731_N0511_R137_T37XEJ_20250917T140915.SAFE | Date: 2025-09-17T14:50:50.000000Z
Sentinel-2 product: S2C_MSIL1C_20250917T110731_N0511_R137_T38XMQ_20250917T133205.SAFE | Date: 2025-09-17T14:07:00.000000Z
Sentinel-2 product: S2C_MSIL1C_20250917T110731_N0511_R137_T39XWM_20250917T133205.SAFE | Date: 2025-09-17T14:06:46.000000Z
Sentinel-2 product: S2C_MSIL1C_20250917T110731_N0511_R137_T39XWL_20250917T133205.SAFE | Date: 2025-09-17T14:07:02.000000Z
Sentinel-2 product: S2C_MSIL2A_20250917T110731_N0511_R137_T41XML_20250917T140915.SAFE | Date: 2025-09-17T14:50:50.000000Z


# 4. Filter by location (bounding box)

In [5]:
bbox = "5.35 51.4, 5.55 51.4, 5.55 51.5, 5.35 51.5, 5.35 51.4"

params = {
    "$top": 5,
    "$orderby": "ContentDate/Start desc",
    "$filter": "Collection/Name eq 'SENTINEL-2' and OData.CSC.Intersects(area=geography'SRID=4326;POLYGON((" + bbox + "))')"
}
r = requests.get(url, headers=headers, params=params)
results = r.json()

for item in results["value"]:
    print("Sentinel-2 product:", item["Name"], "| Date:", item["OriginDate"])

Sentinel-2 product: S2B_MSIL2A_20250916T104619_N0511_R051_T31UFT_20250916T133243.SAFE | Date: 2025-09-16T14:04:55.000000Z
Sentinel-2 product: S2B_MSIL1C_20250916T104619_N0511_R051_T31UFS_20250916T130440.SAFE | Date: 2025-09-16T13:41:10.000000Z
Sentinel-2 product: S2B_MSIL2A_20250916T104619_N0511_R051_T31UFS_20250916T133243.SAFE | Date: 2025-09-16T14:11:02.000000Z
Sentinel-2 product: S2B_MSIL1C_20250916T104619_N0511_R051_T31UFT_20250916T130440.SAFE | Date: 2025-09-16T13:41:39.000000Z
Sentinel-2 product: S2A_MSIL1C_20250913T104651_N0511_R051_T31UFS_20250913T142659.SAFE | Date: 2025-09-13T16:15:24.000000Z


# 5. Filter by tile

In [6]:
params = {
    "$top": 5,
    "$orderby": "ContentDate/Start desc",
    "$filter": "Collection/Name eq 'SENTINEL-2' and Attributes/OData.CSC.StringAttribute/any(att: att/Name eq 'tileId' and att/Value eq '31UFT')"
}

r = requests.get(url, headers=headers, params=params)
results = r.json()

for item in results["value"]:
    print("Sentinel-2 product:", item["Name"], "| Date:", item["OriginDate"])

Sentinel-2 product: S2B_MSIL2A_20250916T104619_N0511_R051_T31UFT_20250916T133243.SAFE | Date: 2025-09-16T14:04:55.000000Z
Sentinel-2 product: S2B_MSIL1C_20250916T104619_N0511_R051_T31UFT_20250916T130440.SAFE | Date: 2025-09-16T13:41:39.000000Z
Sentinel-2 product: S2A_MSIL1C_20250913T104651_N0511_R051_T31UFT_20250913T142659.SAFE | Date: 2025-09-13T16:15:42.000000Z
Sentinel-2 product: S2A_MSIL2A_20250913T104651_N0511_R051_T31UFT_20250913T161813.SAFE | Date: 2025-09-13T17:13:18.000000Z
Sentinel-2 product: S2B_MSIL2A_20250913T103619_N0511_R008_T31UFT_20250913T131830.SAFE | Date: 2025-09-13T13:55:10.000000Z


# 6. Download a product

In [7]:
# always refresh your token before downloading (it expires in 10 minutes!)
token = requests.post(
    "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
    data={
        "grant_type": "client_credentials",
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
    }
).json()
ACCESS_TOKEN = token["access_token"]

product_id = results["value"][0]["Id"]
download_url = f"https://download.dataspace.copernicus.eu/odata/v1/Products({product_id})/$value"

headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}

print("Download link:", download_url)

with requests.get(download_url, headers=headers, stream=True) as r:
    r.raise_for_status()
    with open("sentinel_product.zip", "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

print("✅ Download complete")

Download link: https://download.dataspace.copernicus.eu/odata/v1/Products(471dfd90-2e76-48d4-bdc6-0416b8dd2de3)/$value
✅ Download complete


In [8]:
import os
print("File size (MB):", os.path.getsize("sentinel_product.zip") / 1e6)

File size (MB): 976.506162


# 7. Filter by cloud cover

In [10]:
import requests

token = requests.post(
    "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
    data={
        "grant_type": "client_credentials",
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
    }
).json()
ACCESS_TOKEN = token["access_token"]
headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}

url = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"

params = {
    "$top": 3,
    "$orderby": "ContentDate/Start desc",
    "$filter": (
        "Collection/Name eq 'SENTINEL-2' "
        "and Attributes/OData.CSC.StringAttribute/any(att: att/Name eq 'tileId' and att/Value eq '31UFT') "
        "and Attributes/OData.CSC.DoubleAttribute/any(a:a/Name eq 'cloudCover' and a/OData.CSC.DoubleAttribute/Value lt 30)"
    )
}

r = requests.get(url, headers=headers, params=params)
print("Status:", r.status_code)

if r.ok:
    results = r.json()
    for item in results["value"]:
        print("Product:", item["Name"], "| Date:", item["OriginDate"])
else:
    print(r.text)

Status: 200
Product: S2A_MSIL1C_20250910T104041_N0511_R008_T31UFT_20250910T155709.SAFE | Date: 2025-09-10T17:38:11.000000Z
Product: S2A_MSIL1C_20250821T104041_N0511_R008_T31UFT_20250821T110137.SAFE | Date: 2025-08-21T12:37:46.000000Z
Product: S2C_MSIL2A_20250819T104041_N0511_R008_T31UFT_20250819T155312.SAFE | Date: 2025-08-19T16:49:12.000000Z
